# Rules that change most across groups

## Setup

In [1]:
from pathlib import Path
import sys
import warnings

import pandas as pd

ROOT = next(
    folder for folder in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (folder / "result_summary" / "data_helper.py").exists()
)
sys.path[:0] = [
    str(ROOT / "result_summary"),
    str(ROOT / "result_summary" / "differential_rules" / "scripts"),
]

import data_helper as dh
import differential_stats as ds
import final_figures_vis as fv
import rule_metrics as rm

warnings.filterwarnings("ignore")

RULE_MAX_ITEMS = 2
MIN_FOVS = 10
MIN_FOVS_WITH_RULE = 5
MIN_GAP = 0.15
ORDER = {
    "Pathological score": ["Control", "Mild", "Severe"],
    "Clinical score": ["Control", "Mild", "Severe"],
    "Pathological stage": ["Control", 0, 1, 2, 3],
    "GI stage": ["Control", 0, 1, 2, 3],
    "Cortico Response": ["Control", "Responder", "Non-responder"],
    "Days after Transplant grouped": ["Control", "<30", "30-100", ">100"],
}
KIND_NAMES = {rm.ATTRACTS: "Attraction", rm.AVOIDS: "Avoidance"}
RULE_SCOPE = f"pairs · mining FDR ≤ {dh.MAX_FDR}"

## Data

In [ ]:
df_cells, df_fovs, df_biopsy = dh.load_spatial_data()
df_results = dh.load_results(rule_max_items=RULE_MAX_ITEMS, kind=None)
rules = dh.prepare_rules(df_results)
rules = rules[rules["FOV"].isin(df_fovs["FOV"])]
rule_values = {
    kind: ds.direction_matrix(ds.state_tables(rules, df_cells, df_fovs["FOV"], kinds=(kind,))[2],
                              1 if kind == rm.ATTRACTS else -1)
    for kind in KIND_NAMES
}

Dropped 21 FOVs: Control_S
Loaded 598093 cells; FOVs: 267, biopsies: 73, patients: 71
Read results_CN_pairwise.csv
Loaded 16600 rules.
Kept 9719 rules at FDR <= 0.05.
Kept 9719 rules that add information.
Kept 9647 rules naming no Unidentified.
Kept 9647 rules (max_items=2, kind=None).


## Rule selection

In [ ]:
def group_fovs(organ, trend_col, group):
    meta = df_fovs if organ is None else df_fovs[df_fovs["Organ"] == organ]
    return meta.loc[meta[trend_col] == group, "FOV"].tolist()


def group_counts(organ, trend_col, kind):
    """Rule x group: FOVs with the rule, and FOVs where the rule can be tested."""
    found, tested = {}, {}
    for group in ORDER[trend_col]:
        values = rule_values[kind][group_fovs(organ, trend_col, group)]
        if values.shape[1] < MIN_FOVS:
            continue
        tested[group] = values.notna().sum(axis=1)
        found[group] = values.sum(axis=1).astype(int)
    found, tested = pd.DataFrame(found), pd.DataFrame(tested)
    found["Total"] = found.sum(axis=1)
    tested["Total"] = tested.sum(axis=1)
    return found, tested

In [ ]:
def rule_gaps(found, tested, control_in_gap=False):
    """Change from the first group to the last, for every rule that can be compared:
    enough testable FOVs at both ends, and enough FOVs with the rule at one of them."""
    groups = [g for g in tested.columns[:-1] if control_in_gap or g != "Control"]
    ends = [groups[0], groups[-1]]
    can_compare = (tested[ends].ge(MIN_FOVS).all(axis=1)
                   & found[ends].ge(MIN_FOVS_WITH_RULE).any(axis=1))
    share = found.loc[can_compare, ends] / tested.loc[can_compare, ends]
    return share[ends[1]] - share[ends[0]], *ends

In [ ]:
def large_gaps(gap):
    """Rules whose change is at least MIN_GAP: rises, then drops, biggest first."""
    big = gap[gap.abs() >= MIN_GAP]
    return pd.concat([big[big >= 0].sort_values(ascending=False), big[big < 0].sort_values()])

In [ ]:
def end_fdr(organ, trend_col, kind, first, last, rules):
    """FDR of each rule's change from the first group to the last: permutation test, corrected across `rules`."""
    result = ds.compare(rule_values[kind].loc[rules], group_fovs(organ, trend_col, first),
                        group_fovs(organ, trend_col, last), first, last,
                        min_eligible=MIN_FOVS, min_present=0)
    return result["fdr"]

In [ ]:
def show_rule_change_heatmap(organ, trend_col, kind, control_in_gap=False):
    found, tested = group_counts(organ, trend_col, kind)
    gap, first, last = rule_gaps(found, tested, control_in_gap)
    shown = large_gaps(gap)
    if shown.empty:
        print(f"{organ} · {trend_col} · {kind}: no rule changes by {MIN_GAP:.0%} or more")
        return
    fdr = end_fdr(organ, trend_col, kind, first, last, gap.index)
    fv.plot_rule_change_heatmap(
        found.loc[shown.index], tested.loc[shown.index],
        title=f"Gap by {trend_col}",
        organ=organ or "Colon / Duodenum",
        subtitle=f"{first} → {last}: change ≥ {MIN_GAP:.0%}"
                 + ("" if control_in_gap else " · Control not ranked"),
        params=f"{KIND_NAMES[kind]} · {RULE_SCOPE} · per end ≥{MIN_FOVS} testable FOVs, "
               f"≥{MIN_FOVS_WITH_RULE} with the rule at one end",
        note=f"Cells: found / testable FOVs · FDR: permutation test, BH over {len(fdr)} rules",
        skipped=None if control_in_gap else "Control",
        fdr=fdr,
        classes=dh.rule_class(shown.index, df_cells),
        line_color=fv.KIND_COLORS[kind],
        save=f"rule_change_heatmap_{kind}_{organ or 'both'}_{trend_col.replace(' ', '')}"
             + ("_control" if control_in_gap else ""),
    )

## Colon

In [ ]:
show_rule_change_heatmap("Colon", "Pathological score", kind="attracts", control_in_gap=True)

In [ ]:
show_rule_change_heatmap("Colon", "Pathological score", kind="avoids", control_in_gap=True)

In [ ]:
show_rule_change_heatmap("Colon", "Clinical score", kind="attracts", control_in_gap=True)

In [ ]:
show_rule_change_heatmap("Colon", "Clinical score", kind="avoids", control_in_gap=True)

In [ ]:
show_rule_change_heatmap("Colon", "Pathological stage", kind="attracts", control_in_gap=True)

In [ ]:
show_rule_change_heatmap("Colon", "Pathological stage", kind="avoids", control_in_gap=True)

In [ ]:
show_rule_change_heatmap("Colon", "GI stage", kind="attracts", control_in_gap=True)

In [ ]:
show_rule_change_heatmap("Colon", "GI stage", kind="avoids", control_in_gap=True)

In [ ]:
show_rule_change_heatmap("Colon", "Cortico Response", kind="attracts")

In [ ]:
show_rule_change_heatmap("Colon", "Cortico Response", kind="avoids")

In [ ]:
show_rule_change_heatmap("Colon", "Days after Transplant grouped", kind="attracts")

In [ ]:
show_rule_change_heatmap("Colon", "Days after Transplant grouped", kind="attracts", control_in_gap=True)

In [ ]:
show_rule_change_heatmap("Colon", "Days after Transplant grouped", kind="avoids")

In [ ]:
show_rule_change_heatmap("Colon", "Days after Transplant grouped", kind="avoids", control_in_gap=True)

## Duodenum

In [ ]:
show_rule_change_heatmap("Duodenum", "Pathological score", kind="attracts", control_in_gap=True)

In [ ]:
show_rule_change_heatmap("Duodenum", "Pathological score", kind="avoids", control_in_gap=True)

In [ ]:
show_rule_change_heatmap("Duodenum", "Clinical score", kind="attracts", control_in_gap=True)

In [ ]:
show_rule_change_heatmap("Duodenum", "Clinical score", kind="avoids", control_in_gap=True)

In [ ]:
show_rule_change_heatmap("Duodenum", "Pathological stage", kind="attracts", control_in_gap=True)

In [ ]:
show_rule_change_heatmap("Duodenum", "Pathological stage", kind="avoids", control_in_gap=True)

In [ ]:
show_rule_change_heatmap("Duodenum", "GI stage", kind="attracts", control_in_gap=True)

In [ ]:
show_rule_change_heatmap("Duodenum", "GI stage", kind="avoids", control_in_gap=True)

In [ ]:
show_rule_change_heatmap("Duodenum", "Cortico Response", kind="attracts")

In [ ]:
show_rule_change_heatmap("Duodenum", "Cortico Response", kind="avoids")

In [ ]:
show_rule_change_heatmap("Duodenum", "Days after Transplant grouped", kind="attracts")

In [ ]:
show_rule_change_heatmap("Duodenum", "Days after Transplant grouped", kind="attracts", control_in_gap=True)

In [ ]:
show_rule_change_heatmap("Duodenum", "Days after Transplant grouped", kind="avoids")

In [ ]:
show_rule_change_heatmap("Duodenum", "Days after Transplant grouped", kind="avoids", control_in_gap=True)